# Demo: Torus MASCAF

Fit a cable morphology to the torus demo mesh, using the same tip extension and surface-area scaling as the other demos. The torus skeleton is already a closed loop, so tip extension does not add nodes.

In [6]:
from mascaf import *

import logging

logging.basicConfig(level=logging.INFO)

## Mesh and skeleton

Load the demo mesh and its curve skeleton. The skeleton is the centerline the cable fit follows.

In [7]:
mm = MeshManager(mesh_path="../../data/demo/torus.obj")
mm.print_mesh_analysis()
skeleton = SkeletonGraph.from_txt("../../data/demo/torus.polylines.txt")
mesh_fig, camera = visualize_mesh_3d(
    mesh=mm,
    title="Torus",
    skel=skeleton,
    show_axes=False,
    return_camera=True,
)
mesh_fig

INFO:mascaf.mesh:Loaded mesh: 512 vertices, 1024 faces
INFO:mascaf.mesh:Mesh Analysis Report
INFO:mascaf.mesh:====================
INFO:mascaf.mesh:
Geometry:
INFO:mascaf.mesh:  * Vertices: 512
INFO:mascaf.mesh:  * Faces: 1024
INFO:mascaf.mesh:  * Components: 1
INFO:mascaf.mesh:  * Volume: 76.45
INFO:mascaf.mesh:  * Bounds: [-5.0, -1.0, -5.0] to [5.0, 1.0, 5.0]
INFO:mascaf.mesh:
Mesh Quality:
INFO:mascaf.mesh:  * Watertight: True
INFO:mascaf.mesh:  * Winding Consistent: True
INFO:mascaf.mesh:  * Normal Direction: outward
INFO:mascaf.mesh:  * Duplicate Vertices: 0
INFO:mascaf.mesh:  * Degenerate Faces: 0
INFO:mascaf.mesh:
Topology:
INFO:mascaf.mesh:  * Genus: 1
INFO:mascaf.mesh:  * Euler Characteristic: 0
INFO:mascaf.mesh:
No issues detected
INFO:mascaf.mesh:
Recommendation:
INFO:mascaf.mesh:  Mesh appears to be in good condition.
INFO:mascaf.mesh:====================


## Cable fit

`CableFitter` resamples the skeleton so edges are at most `max_edge_length` and estimates a radius at each sample from the mesh cross-section.

In [8]:
max_edge_length = 1.0

morph = CableFitter(FitOptions(max_edge_length=max_edge_length)).fit(mm.mesh, skeleton)

INFO:mascaf.cable_fitting:Starting cable fit with 32 skeleton nodes, 32 skeleton edges, 512 mesh vertices, and max_edge_length=1.0
INFO:mascaf.morphology_graph:Resampled morphology: 25 nodes, 25 edges (max_edge_length=1.0000, source 32 nodes / 32 edges, 1 sections)
INFO:mascaf.cable_fitting:Computing node radii: strategy=equivalent_area, 25 nodes, probe_eps=1.00e-04, tries=3
INFO:mascaf.cable_fitting:Radii computed: min=0.988331 max=0.992011 mean=0.989607 median=0.988972
INFO:mascaf.cable_fitting:Finished cable fit with 25 morphology nodes and 25 morphology edges


## Fitted cable

The mesh and skeleton are on the left; the cable model is on the right. Both panels use the same view. Validation compares cable volume and surface area with the mesh.

In [9]:
fig = visualize_mesh_cable_3d(mm, skeleton, morph, show_axes=False, camera=camera)
fig.show()

validator = Validation(mm, skeleton, morph)
validator.full_validation()

INFO:swctools.geometry:batch_frusta count=25 sides=16 end_caps=False verts=800 faces=800
INFO:swctools.geometry:FrustaSet.from_swc_model edges=25 sides=16 end_caps=False
INFO:swctools.viz:plot_model slider=False frusta=25 show_frusta=True show_centroid=True


INFO:mascaf.validation:Initialized Validation from MorphologyGraph
INFO:mascaf.validation:  Mesh: 512 vertices, 1024 faces
INFO:mascaf.validation:  Skeleton: 32 nodes, 32 edges
INFO:mascaf.validation:  MorphologyGraph: 25 nodes, 25 edges
INFO:mascaf.validation:No branch vertices (degree > 2) found; overlap correction has no effect. Showing account_for_overlaps=False only.
INFO:mascaf.validation:Validation Results, account_for_overlaps=False:
INFO:mascaf.validation:-- Volume Comparison:
INFO:mascaf.validation:---- Mesh volume:       76.4496
INFO:mascaf.validation:---- Morphology volume: 76.4805
INFO:mascaf.validation:---- Ratio:             1.0004
INFO:mascaf.validation:---- Error:             0.0309
INFO:mascaf.validation:---- Relative error:    0.04%
INFO:mascaf.validation:-- Surface Area Comparison:
INFO:mascaf.validation:---- Mesh area:         156.2721
INFO:mascaf.validation:---- Morphology area:   154.5675
INFO:mascaf.validation:---- Ratio:             0.9891
INFO:mascaf.validatio

{'n_outside_nodes': 0, 'n_crossing_edges': 0, 'n_fully_outside_edges': 0}

## Surface-area normalization

Scale every radius by one factor so the cable surface area matches the mesh.

In [10]:
morph.scale_radii_to_match_mesh(
    mm.mesh, metric="surface_area", account_for_overlaps=False
)
fig = visualize_mesh_cable_3d(mm, skeleton, morph, show_axes=False, camera=camera, show_nodes=True, node_size=8, centroid_line_width=5)
fig.show()

validator = Validation(mm, skeleton, morph)
validator.full_validation()

INFO:swctools.geometry:batch_frusta count=25 sides=16 end_caps=False verts=800 faces=800
INFO:swctools.geometry:FrustaSet.from_swc_model edges=25 sides=16 end_caps=False
INFO:swctools.viz:plot_model slider=False frusta=25 show_frusta=True show_centroid=True


INFO:mascaf.validation:Initialized Validation from MorphologyGraph
INFO:mascaf.validation:  Mesh: 512 vertices, 1024 faces
INFO:mascaf.validation:  Skeleton: 32 nodes, 32 edges
INFO:mascaf.validation:  MorphologyGraph: 25 nodes, 25 edges
INFO:mascaf.validation:No branch vertices (degree > 2) found; overlap correction has no effect. Showing account_for_overlaps=False only.
INFO:mascaf.validation:Validation Results, account_for_overlaps=False:
INFO:mascaf.validation:-- Volume Comparison:
INFO:mascaf.validation:---- Mesh volume:       76.4496
INFO:mascaf.validation:---- Morphology volume: 78.1766
INFO:mascaf.validation:---- Ratio:             1.0226
INFO:mascaf.validation:---- Error:             1.7270
INFO:mascaf.validation:---- Relative error:    2.26%
INFO:mascaf.validation:-- Surface Area Comparison:
INFO:mascaf.validation:---- Mesh area:         156.2721
INFO:mascaf.validation:---- Morphology area:   156.2721
INFO:mascaf.validation:---- Ratio:             1.0000
INFO:mascaf.validatio

{'n_outside_nodes': 0, 'n_crossing_edges': 0, 'n_fully_outside_edges': 0}